# Static and movie RDM trajectories in the static PCA basis

Compute one stimulus RDM at every timepoint for both matched conditions. Fit three principal components to the static RDM time series, then project the movie RDM time series into exactly the same stimulus-pair basis. Static time 0 s is color-aligned to movie time 2.5 s, so the extra movie interval from 2.0 to 2.5 s appears in the earlier part of the colormap.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import numpy as np
from sklearn.decomposition import PCA
import yaml

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "config.yaml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find config.yaml above the notebook directory.")
    # end if PROJECT_ROOT.parent
    PROJECT_ROOT = PROJECT_ROOT.parent
# end while not config.yaml exists

ENV = os.getenv("MY_ENV", "tiziano_mac_mini")
with open(PROJECT_ROOT / "config.yaml", "r") as file:
    config = yaml.safe_load(file)
# end with open

paths = config[ENV]["paths"]
sys.path.extend([paths["src_path"], paths["useful_stuff_path"]])

from project_specific_utils import (
    compute_rdm_timeseries, load_natraster, match_static_dynamic_rasters,
)
from useful_stuff.general_utils import TimeSeries

In [ ]:
@dataclass
class Cfg:
    static_exp_name: str = "red_20260726to27"
    dynamic_exp_name: str = "red_20260720to24"
    source_fs: int = 1000
    plot_fs: int = 100
    rdm_metric: str = "cosine_cnt"
    n_components: int = 3
    movie_window_s: tuple[float, float] = (2.0, 3.5)
    movie_pca_start_s: float = 0.15
    static_alignment_s: float = 2.5
    cmap: str = "viridis"


cfg = Cfg()
data_dir = Path(paths["data_path"]) / "data"
static_path = data_dir / f"{cfg.static_exp_name}_natraster_img.mat"
movie_path = data_dir / f"{cfg.dynamic_exp_name}_natraster_vid.mat"
cfg, static_path, movie_path

## Load, match, and compute time-resolved RDMs

Matching retains the standard `img_` response for every movie identity and puts the two conditions in the same stimulus order. After temporal resampling, each timepoint is converted from a channels-by-stimuli response matrix to the vectorized upper triangle of its stimulus RDM. Thus, PCA samples are timepoints and PCA features are the same stimulus pairs in both conditions.

In [ ]:
static_rasters, static_names = load_natraster(static_path)
movie_rasters, movie_names = load_natraster(movie_path)
static_rasters, movie_rasters, shared_stimuli = match_static_dynamic_rasters(
    static_rasters, static_names, movie_rasters, movie_names,
)

# Use the established resampling utility on channels x time x stimuli arrays.
static_ts = TimeSeries(static_rasters, cfg.source_fs)
movie_ts = TimeSeries(movie_rasters, cfg.source_fs)
static_ts.resample(cfg.plot_fs)
movie_ts.resample(cfg.plot_fs)

# Each PCA sample is a timepoint and each feature is one matched stimulus pair.
static_rdms = compute_rdm_timeseries(static_ts.get_array(), cfg.rdm_metric)
movie_rdms = compute_rdm_timeseries(movie_ts.get_array(), cfg.rdm_metric)
static_times_s = np.arange(static_rdms.shape[0]) / cfg.plot_fs
movie_times_s = np.arange(movie_rdms.shape[0]) / cfg.plot_fs
movie_mask = (movie_times_s >= cfg.movie_window_s[0]) & (movie_times_s < cfg.movie_window_s[1])
movie_pca_mask = movie_times_s >= cfg.movie_pca_start_s

print(f"Matched stimuli: {len(shared_stimuli)}")
print(f"Static RDM time series: {static_rdms.shape} (time, stimulus pairs)")
print(f"Movie RDM time series: {movie_rdms.shape} (time, stimulus pairs)")
print(f"Movie window: {movie_mask.sum()} timepoints")
print(f"Movie PCA fit: {movie_pca_mask.sum()} timepoints ({cfg.movie_pca_start_s:.2f} s to end)")

## Fit on static RDMs and project movie RDMs

`PCA.transform` applies both the static-derived stimulus-pair centering and the static-derived component directions to the movie RDM time series.

In [ ]:
static_pca = PCA(n_components=cfg.n_components)
static_scores = static_pca.fit_transform(static_rdms)
movie_scores = static_pca.transform(movie_rdms[movie_mask])
movie_window_times_s = movie_times_s[movie_mask]

print("Explained variance: " + ", ".join(
    f"PC{index}: {ratio:.1%}"
    for index, ratio in enumerate(static_pca.explained_variance_ratio_, start=1)
))
print(
    f"Cumulative variance explained by PC1–PC{cfg.n_components}: "
    f"{static_pca.explained_variance_ratio_.sum():.1%}"
)

## 3D RDM trajectories

The static color coordinates are shifted by 2.5 s. Consequently, static time 0 s and movie time 2.5 s receive exactly the same color.

In [ ]:
static_color_times_s = static_times_s + cfg.static_alignment_s
color_norm = Normalize(*cfg.movie_window_s)

figure = plt.figure(figsize=(14, 6), constrained_layout=True)
axes = [
    figure.add_subplot(1, 2, 1, projection="3d"),
    figure.add_subplot(1, 2, 2, projection="3d"),
]

for axis, scores, color_times_s, title in (
        (axes[0], static_scores, static_color_times_s, "Static RDM: 0–1 s"),
        (axes[1], movie_scores, movie_window_times_s, "Movie RDM: 2–3.5 s"),
        ):
    # Give every adjacent pair of 3D points the color of its midpoint time.
    segments = np.stack([scores[:-1, :3], scores[1:, :3]], axis=1)
    segment_times_s = (color_times_s[:-1] + color_times_s[1:]) / 2
    trajectory = Line3DCollection(
        segments, cmap=cfg.cmap, norm=color_norm, linewidth=3,
    )
    trajectory.set_array(segment_times_s)
    axis.add_collection3d(trajectory)
    axis.set(
        xlabel="Static RDM PC 1", ylabel="Static RDM PC 2",
        zlabel="Static RDM PC 3", title=title,
    )
    axis.set_box_aspect((1, 1, 1))
# end for axis, scores

# Identical limits make the two trajectories directly comparable in the shared basis.
all_scores = np.vstack([static_scores[:, :3], movie_scores[:, :3]])
for dimension, setter in enumerate(("set_xlim", "set_ylim", "set_zlim")):
    lower, upper = all_scores[:, dimension].min(), all_scores[:, dimension].max()
    margin = 0.05 * (upper - lower)
    for axis in axes:
        getattr(axis, setter)(lower - margin, upper + margin)
    # end for axis
# end for dimension, setter

colorbar = figure.colorbar(trajectory, ax=axes, shrink=0.75, pad=0.08)
colorbar.set_label("Movie-aligned time (s) — static onset = 2.5 s")
colorbar.set_ticks([2.0, 2.5, 3.0, 3.5])
plt.show()

## PC1–PC2 RDM overlay

Both conditions are drawn on the same two-dimensional axes. Point colors retain the movie-aligned time scale used above; line style distinguishes the trajectories where they overlap.

In [ ]:
figure, axis = plt.subplots(figsize=(8, 7), constrained_layout=True)

axis.plot(
    static_scores[:, 0], static_scores[:, 1],
    color="tab:blue", linewidth=1.5, label="Static RDM: 0–1 s", zorder=1,
)
axis.plot(
    movie_scores[:, 0], movie_scores[:, 1],
    color="tab:orange", linewidth=1.5, linestyle="--",
    label="Movie RDM: 2–3.5 s", zorder=1,
)
axis.scatter(
    static_scores[:, 0], static_scores[:, 1],
    c=static_color_times_s, cmap=cfg.cmap, norm=color_norm,
    marker="o", s=28, edgecolor="none", zorder=2,
)
points = axis.scatter(
    movie_scores[:, 0], movie_scores[:, 1],
    c=movie_window_times_s, cmap=cfg.cmap, norm=color_norm,
    marker="^", s=30, edgecolor="none", zorder=2,
)

axis.set(
    xlabel=f"Static RDM PC 1 ({static_pca.explained_variance_ratio_[0]:.1%})",
    ylabel=f"Static RDM PC 2 ({static_pca.explained_variance_ratio_[1]:.1%})",
    title=f"Static and movie RDM trajectories ({cfg.rdm_metric})",
)
axis.set_aspect("equal", adjustable="box")
axis.legend()
colorbar = figure.colorbar(points, ax=axis, pad=0.02)
colorbar.set_label("Movie-aligned time (s) — static onset = 2.5 s")
colorbar.set_ticks([2.0, 2.5, 3.0, 3.5])
plt.show()

# Static and movie RDM trajectories in the movie PCA basis

Fit three principal components to the movie RDM time series from 150 ms through the final sample. Then project the complete static RDM trajectory and the displayed 2–3.5 s movie RDM trajectory using the movie-derived stimulus-pair centering and component directions.

In [ ]:
# Fit the movie basis on every movie RDM sample from 150 ms onward.
movie_pca = PCA(n_components=cfg.n_components)
movie_pca.fit(movie_rdms[movie_pca_mask])

# Apply the movie-derived centering and axes to both displayed RDM trajectories.
static_in_movie_scores = movie_pca.transform(static_rdms)
movie_in_movie_scores = movie_pca.transform(movie_rdms[movie_mask])

print("Movie-basis explained variance: " + ", ".join(
    f"PC{index}: {ratio:.1%}"
    for index, ratio in enumerate(movie_pca.explained_variance_ratio_, start=1)
))
print(
    f"Cumulative variance explained by movie PC1–PC{cfg.n_components}: "
    f"{movie_pca.explained_variance_ratio_.sum():.1%}"
)

## 3D RDM trajectories in the movie basis

The time alignment, colormap, and displayed intervals are identical to the static-basis plot above; only the PCA basis changes.

In [ ]:
figure = plt.figure(figsize=(14, 6), constrained_layout=True)
axes = [
    figure.add_subplot(1, 2, 1, projection="3d"),
    figure.add_subplot(1, 2, 2, projection="3d"),
]

for axis, scores, color_times_s, title in (
        (axes[0], static_in_movie_scores, static_color_times_s, "Static RDM: 0–1 s"),
        (axes[1], movie_in_movie_scores, movie_window_times_s, "Movie RDM: 2–3.5 s"),
        ):
    # Give every adjacent pair of 3D points the color of its midpoint time.
    segments = np.stack([scores[:-1, :3], scores[1:, :3]], axis=1)
    segment_times_s = (color_times_s[:-1] + color_times_s[1:]) / 2
    trajectory = Line3DCollection(
        segments, cmap=cfg.cmap, norm=color_norm, linewidth=3,
    )
    trajectory.set_array(segment_times_s)
    axis.add_collection3d(trajectory)
    axis.set(
        xlabel="Movie RDM PC 1", ylabel="Movie RDM PC 2",
        zlabel="Movie RDM PC 3", title=title,
    )
    axis.set_box_aspect((1, 1, 1))
# end for axis, scores

# Identical limits make the two trajectories directly comparable in the shared basis.
all_movie_basis_scores = np.vstack([
    static_in_movie_scores[:, :3], movie_in_movie_scores[:, :3],
])
for dimension, setter in enumerate(("set_xlim", "set_ylim", "set_zlim")):
    lower = all_movie_basis_scores[:, dimension].min()
    upper = all_movie_basis_scores[:, dimension].max()
    margin = 0.05 * (upper - lower)
    for axis in axes:
        getattr(axis, setter)(lower - margin, upper + margin)
    # end for axis
# end for dimension, setter

colorbar = figure.colorbar(trajectory, ax=axes, shrink=0.75, pad=0.08)
colorbar.set_label("Movie-aligned time (s) — static onset = 2.5 s")
colorbar.set_ticks([2.0, 2.5, 3.0, 3.5])
plt.show()

## PC1–PC2 RDM overlay in the movie basis

As above, line style identifies condition and point color gives movie-aligned time.

In [ ]:
figure, axis = plt.subplots(figsize=(8, 7), constrained_layout=True)

axis.plot(
    static_in_movie_scores[:, 0], static_in_movie_scores[:, 1],
    color="tab:blue", linewidth=1.5, label="Static RDM: 0–1 s", zorder=1,
)
axis.plot(
    movie_in_movie_scores[:, 0], movie_in_movie_scores[:, 1],
    color="tab:orange", linewidth=1.5, linestyle="--",
    label="Movie RDM: 2–3.5 s", zorder=1,
)
axis.scatter(
    static_in_movie_scores[:, 0], static_in_movie_scores[:, 1],
    c=static_color_times_s, cmap=cfg.cmap, norm=color_norm,
    marker="o", s=28, edgecolor="none", zorder=2,
)
points = axis.scatter(
    movie_in_movie_scores[:, 0], movie_in_movie_scores[:, 1],
    c=movie_window_times_s, cmap=cfg.cmap, norm=color_norm,
    marker="^", s=30, edgecolor="none", zorder=2,
)

axis.set(
    xlabel=f"Movie RDM PC 1 ({movie_pca.explained_variance_ratio_[0]:.1%})",
    ylabel=f"Movie RDM PC 2 ({movie_pca.explained_variance_ratio_[1]:.1%})",
    title=f"Static and movie RDM trajectories in the movie basis ({cfg.rdm_metric})",
)
axis.set_aspect("equal", adjustable="box")
axis.legend()
colorbar = figure.colorbar(points, ax=axis, pad=0.02)
colorbar.set_label("Movie-aligned time (s) — static onset = 2.5 s")
colorbar.set_ticks([2.0, 2.5, 3.0, 3.5])
plt.show()

# Correlation between the static and movie RDM PCA bases

Each cell is the signed Pearson correlation across stimulus pairs between one static loading vector and one movie loading vector. PCA component signs are arbitrary, so correlation magnitude measures alignment while the sign only records the orientation selected by each fit. The full matrix allows components with different rank orders to be compared.

In [ ]:
# Stack both sets of stimulus-pair loading vectors and retain the cross-block.
all_components = np.vstack([static_pca.components_, movie_pca.components_])
basis_correlation = np.corrcoef(all_components)[
    :cfg.n_components, cfg.n_components:
]

print("Static-RDM-PC rows × movie-RDM-PC columns (signed Pearson r):")
print(np.array2string(basis_correlation, precision=3, suppress_small=True))

figure, axis = plt.subplots(figsize=(6.5, 5.5), constrained_layout=True)
image = axis.imshow(basis_correlation, cmap="coolwarm", vmin=-1, vmax=1)

for static_index in range(cfg.n_components):
    for movie_index in range(cfg.n_components):
        correlation = basis_correlation[static_index, movie_index]
        text_color = "white" if abs(correlation) >= 0.55 else "black"
        axis.text(
            movie_index, static_index, f"{correlation:.2f}",
            ha="center", va="center", color=text_color, fontsize=12,
        )
    # end for movie_index
# end for static_index

component_indices = np.arange(cfg.n_components)
axis.set(
    xticks=component_indices, yticks=component_indices,
    xticklabels=[f"Movie RDM PC {index}" for index in range(1, cfg.n_components + 1)],
    yticklabels=[f"Static RDM PC {index}" for index in range(1, cfg.n_components + 1)],
    xlabel="Movie RDM PCA basis (fit from 150 ms to end)",
    ylabel="Static RDM PCA basis",
    title="Pairwise correlation of PCA stimulus-pair loadings",
)
colorbar = figure.colorbar(image, ax=axis, pad=0.03)
colorbar.set_label("Pearson r")
plt.show()